In [0]:
from pyspark.sql.types import *

schema=StructType([
    StructField("emp_id", IntegerType(),True),
    StructField("emp_name", StringType(),True),
    StructField("age", IntegerType(),True),
    StructField("department", StringType()),
    StructField("salary", IntegerType(),True),
    StructField("city", StringType(),True),
])

df_csv=(spark.read
.option("header","true")
.schema(schema)
.option("mode","PERMISSIVE")
.csv("/Volumes/workspace/default/pyspark_files_volume/employees.csv")
)
df_csv.show()

df_csv.printSchema()

In [0]:

from pyspark.sql.types import *

schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("city", StringType(), True),

    StructField(
        "address",
        StructType([
            StructField("street", StringType(), True),
            StructField("state", StringType(), True),
            StructField("pincode", IntegerType(), True)
        ]),
        True
    ),

    StructField(
        "skills",
        ArrayType(StringType()),
        True
    ),

    StructField(
        "project",
        StructType([
            StructField("project_id", StringType(), True),
            StructField("project_name", StringType(), True)
        ]),
        True
    )
])


df_json=(
    spark.read
    .option("multiline","true")
    .option("header","true")
    .schema(schema)
    .option("mode", "PERMISSIVE")
    .json("/Volumes/workspace/default/pyspark_files_volume/employees.json")
)

df_json.show()
df_json.printSchema()





In [0]:
df_csv.write.mode("overwrite").parquet("/Volumes/workspace/default/pyspark_files_volume/employees_parquet")
df_parquet=spark.read.parquet("/Volumes/workspace/default/pyspark_files_volume/employees_parquet")
df_parquet.show()
df_parquet.printSchema()



In [0]:

df_parquet.count()
df_parquet.select("emp_name","salary").show(10)
df_parquet.filter(df_parquet.salary>50000).show()
df_parquet.filter(df_parquet.salary>50000).explain()

In [0]:
delta_path="/Volumes/workspace/default/pyspark_files_volume/employees_delta"
df_csv.write.format("delta").mode("overwrite").save(delta_path)
df_delta=spark.read.format("delta").load(delta_path)
df_delta.show()
df_delta.count()

df_delta.printSchema()
display(dbutils.fs.ls(delta_path))
display(dbutils.fs.ls(delta_path + "/_delta_log"))

In [0]:
from pyspark.sql.types import *


schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("city", StringType(), True)
])

new_data=[(1001,"Ravi",35,"Sales",50000,"Bangalore"),
          (1002,"Rajesh",30,"Sales",55000,"Bangalore"),]

new_df=spark.createDataFrame(new_data,schema=schema)
new_df.write.mode("append").format("delta").save(delta_path)

df_delta.show()
df_delta.count()



In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("new_department", StringType(), True)
])

wrong_df=spark.createDataFrame([(123,"anam","IT")],schema=schema)
wrong_df.printSchema()
wrong_df.write.mode("append").format("delta").save(delta_path)
df_delta.show()

df_delta.count()

In [0]:
wrong_df.write.mode("append").option("mergeSchema","true").format("delta").save(delta_path)
wrong_df.printSchema()
wrong_df.show()


In [0]:
old_df=(
    spark.read.format("delta").option("versionAsOf",0).load(delta_path)
)
old_df.show()



In [0]:
spark.sql("""
DESCRIBE HISTORY delta.`/Volumes/workspace/default/pyspark_files_volume/employees_delta`
""")